# 📊 IBM HR Analytics: Minimizing Employee Attrition & Quantifying Financial Loss

> **Author:** Manoj Kumar. V  
> **Role:** Data Analyst  
> **Dataset:** IBM HR Analytics Employee Attrition & Performance (Kaggle) — 1,470 records  
> **Tools:** Python · Pandas · Matplotlib · Seaborn · Power BI · Google Colab  
> **Focus:** Operations & Cost Reduction  

---

### Project Objective

Analyze 1,470 employee records to uncover the primary drivers of attrition at a fictional IBM subsidiary, quantify the financial cost of talent loss by department, and deliver data-backed retention strategies that a Chief Human Resources Officer (CHRO) can act on immediately to reduce turnover and protect corporate capital.

---


## Phase 1 — Environment Setup & Data Loading

*In this opening phase, we configure our data science environment. We import our core analytical packages, suppress administrative runtime warnings, establish a persistent link to Google Drive to secure our data pipeline, and define a consistent, executive-ready design aesthetic for our statistical charts.*

> **Note (Google Colab users):** This notebook mounts Google Drive to retrieve the dataset. If running locally, place the dataset in a data/ folder and use pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv').


In [ ]:
# Mount Google Drive to access our dataset
from google.colab import drive
drive.mount('/content/drive')

# Importing our core analytics libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings to keep code outputs neat
warnings.filterwarnings('ignore')

# Global chart styling for professional, executive-ready visuals
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print("✅ Environment configured and libraries imported successfully!")


In [ ]:
# Load the raw dataset from Google Drive
df = pd.read_csv("/content/drive/MyDrive/HR_Project/WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Preview the first 5 rows to confirm loading worked
print(f"✅ Dataset loaded — {df.shape[0]:,} rows × {df.shape[1]} columns")

df.head()


## Phase 2 — Exploratory Data Inspection (Health Check)

*Before modifying our variables, we inspect the size, data types, and null counts of our dataset to understand the physical and logical structure of our employee files.*


In [ ]:
# How big is our dataset?
print(f"📐 Dataset Dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n--- Data Information ---")

# df.info() gives us column names, data types, and missing value counts
df.info()


In [ ]:
# Check for missing values
print("\n❓ Missing Values Per Column:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

# Show only columns that actually have missing values (if any)
print(missing_summary[missing_summary['Missing Count'] > 0])

# Baseline Attrition summary (clean and flat Pandas output)
print("\n📊 Attrition Baseline Counts:")
print(df['Attrition'].value_counts())

print("\n📊 Attrition Baseline Percentages:")
print((df['Attrition'].value_counts(normalize=True) * 100).round(1))


## Phase 3 — Data Cleaning & Preprocessing

*Real-world corporate databases are rarely analytical-ready. In this phase, we transform our raw HR dataset into a polished, high-performing analytical asset through targeted pruning, numeric mapping, and noise reduction.*


### Step 3.1 — Removing Irrelevant Columns (Noise Reduction)

*We systematically drop variables that contain zero statistical variance (like 'EmployeeCount', 'StandardHours') to keep our data models lean and fast.*


In [ ]:
columns_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']

# Drop the columns permanently from our table
df.drop(columns_to_drop, axis=1, inplace=True)

# Print the new dimensions of our cleaned dataset
print(f"✅ Useless columns removed. New dimensions: {df.shape[0]:,} employees × {df.shape[1]} attributes")


### Step 3.2 — Converting 'Attrition' to a Numeric Field

*We map text-based attrition flags into a binary numeric format (1/0) to enable fast statistical calculations, and prepare the dataset for backend metric queries.*


In [ ]:
# We map 'Yes' to 1 and 'No' to 0 and save it in a new column called 'attrition_num'
# This lets us calculate average attrition rates using simple mathematical means
df['attrition_num'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Verify that the new column was created correctly by checking its unique values
print("\n✅ New numeric attrition column created:")
print(df[['Attrition', 'attrition_num']].head())


### Step 3.3 — Cleaned Dataset Export

*To establish our single source of truth (SSOT) and ensure consistency, we programmatically write our polished, engineered dataset back to Google Drive to drive our interactive Power BI Dashboard.*


In [ ]:
# Export the cleaned DataFrame to a CSV file in your Google Drive folder
# index=False ensures we don't save the row numbers as a separate column!
df.to_csv('/content/drive/MyDrive/HR_Project/ibm_hr_cleaned_data.csv', index=False)

print("🎉 Cleaned dataset successfully exported for Power BI!")


## Phase 4 — Exploratory Data Analysis (EDA)

*In this phase, we execute targeted queries to uncover attrition drivers and calculate the exact financial cost of turnover, translating raw HR data into business intelligence.*


### Step 4.1 — Analyzing Attrition by Department

*We calculate the overall attrition rate across the entire company using our new numeric column. Then, we group our employees by department to see which division is losing the most talent. This shows the CHRO exactly where the "talent leak" is happening.*


In [ ]:
# 1. Calculate the overall attrition rate across the whole company
# Taking the average (mean) of a 1 and 0 column automatically gives us the rate!
overall_attrition_rate = df['attrition_num'].mean() * 100
print(f"Overall Company Attrition Rate: {overall_attrition_rate:.1f}%")

# 2. Group by Department and calculate total employees and attrition rate
dept_stats = df.groupby('Department').agg(
    total_employees=('Attrition', 'count'),
    attrition_rate=('attrition_num', 'mean')
)

# 3. Convert the attrition rate decimal to a clean percentage and sort the table
dept_stats['attrition_rate'] = (dept_stats['attrition_rate'] * 100).round(1)
dept_stats = dept_stats.sort_values(by='attrition_rate', ascending=False)

print("\n--- ATTRITION BY DEPARTMENT ---")
print(dept_stats)


### Step 4.2 — Income & Demographics Analysis (Salary & Age)

*We compare the monthly salaries and ages of employees who left versus those who stayed to see if low pay or age differences are driving people to quit.*


In [ ]:
# 1. Compare the average monthly income of employees who stayed (No) vs left (Yes)
print("\n--- AVERAGE MONTHLY INCOME BY ATTRITION ---")
income_compare = df.groupby('Attrition')['MonthlyIncome'].mean().round(2)
print(income_compare)

# 2. Compare the average age of employees who stayed (No) vs left (Yes)
print("\n--- AVERAGE AGE BY ATTRITION ---")
age_compare = df.groupby('Attrition')['Age'].mean().round(1)
print(age_compare)


### Step 4.3 — Work-Life Balance & Overtime Analysis

*We compare the attrition rates of employees who work overtime against those who do not, and analyze how different work-life balance scores impact their decision to leave. This helps us see if poor work-life balance and extreme workloads are major drivers of employee turnover.*


In [ ]:
# 1. Compare the attrition rate for employees who work Overtime vs No Overtime
print("\n--- ATTRITION RATE BY OVERTIME STATUS ---")
ot_compare = df.groupby('OverTime')['attrition_num'].mean() * 100
print(ot_compare.round(1))

# 2. Compare the attrition rate across Work-Life Balance scores (1 = Bad, 4 = Best)
print("\n--- ATTRITION RATE BY WORK-LIFE BALANCE SCORE ---")
wlb_compare = df.groupby('WorkLifeBalance')['attrition_num'].mean() * 100
print(wlb_compare.round(1))


### Step 4.4 — Calculating the Financial Cost of Attrition

*We translate employee turnover into concrete financial numbers by calculating the cost to replace the departed talent. Using the industry-standard replacement cost of 1.5x an employee's annual salary, we show the CHRO the true economic impact of losing people.*


In [ ]:
# 1. Filter the dataset to get only employees who left (attrition_num == 1)
departed_employees = df[df['attrition_num'] == 1]

# 2. Calculate the average monthly salary of departed employees
avg_departed_salary = departed_employees['MonthlyIncome'].mean()

# 3. Industry standard replacement cost factor is 1.5x the annual salary
# Total annual attrition loss = Number of departed employees * average monthly salary * 1.5 replacement factor
total_departed_count = len(departed_employees)
multiplier = 1.5
total_attrition_cost = total_departed_count * avg_departed_salary * multiplier * 12

print("\n--- FINANCIAL COST OF ATTRITION ---")
print(f"Total Employees Lost: {total_departed_count}")
print(f"Average Monthly Salary of Departed Talent: ₹{avg_departed_salary:.2f}")
print(f"Estimated Annual Attrition Replacement Cost (1.5x salary multiplier): ₹{total_attrition_cost:,.2f}")


## Phase 5 — Data Visualization (Matplotlib & Seaborn)

*In this phase, we convert our raw data findings into highly readable charts. These visualizations make it easy for executives to instantly see the biggest drivers of employee departures.*


### Step 5.1 — Visualizing Department Attrition & Salary Impact

*We build side-by-side charts to compare our findings from Steps 4.1 and 4.2. The left chart shows the attrition rate across different departments, and the right chart compares the average monthly income of employees who left versus those who stayed.*


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure our clean Seaborn whitegrid style is applied
sns.set_theme(style="whitegrid")

# 1. Prepare our data from our earlier aggregations
plot_dept_stats = dept_stats.reset_index()
plot_income_compare = income_compare.reset_index()

# 2. Create a 1-row, 2-column subplot canvas
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))

# --- PLOT 1 (Left): Attrition Rate by Department ---
sns.barplot(
    data=plot_dept_stats,
    x='attrition_rate',
    y='Department',
    ax=axes[0],
    palette='crest',
    hue='Department',
    legend=False
)
axes[0].set_title('1. Employee Attrition Rate by Department', fontsize=13, fontweight='bold', pad=15)
axes[0].set_xlabel('Attrition Percentage (%)', fontsize=11)
axes[0].set_ylabel('Department Name', fontsize=11)
axes[0].set_xlim(0, 35) # Give the bars some breathing room on the right scale

# --- PLOT 2 (Right): Average Monthly Income vs Attrition ---
sns.barplot(
    data=plot_income_compare,
    x='Attrition',
    y='MonthlyIncome',
    ax=axes[1],
    palette='muted',
    hue='Attrition',
    legend=False
)
axes[1].set_title('2. Average Monthly Income vs Attrition', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('Did the Employee Leave?', fontsize=11)
axes[1].set_ylabel('Average Monthly Income (₹)', fontsize=11)

# Prevent title/label overlaps and display the first master chart
plt.tight_layout()
plt.savefig('department_salary_impact.png', dpi=300, bbox_inches='tight')
plt.show()


### Step 5.2 — Visualizing Overtime & Work-Life Balance Drivers

*To visualize our operational insights from Step 4.3, we build another set of side-by-side charts. The left chart shows the dramatic impact of overtime on employee departures, and the right chart shows how attrition rates drop as employee work-life balance scores improve.*


In [ ]:
# 1. Prepare our data from our earlier aggregations
plot_ot_compare = ot_compare.reset_index()
plot_ot_compare.columns = ['OverTime', 'Attrition_Rate']

plot_wlb_compare = wlb_compare.reset_index()
plot_wlb_compare.columns = ['WorkLifeBalance', 'Attrition_Rate']

# 2. Create a second 1-row, 2-column subplot canvas
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))

# --- PLOT 3 (Left): Overtime vs Attrition ---
sns.barplot(
    data=plot_ot_compare,
    x='OverTime',
    y='Attrition_Rate',
    ax=axes[0],
    palette='flare',
    hue='OverTime',
    legend=False
)
axes[0].set_title('3. Attrition Rate by Overtime Status', fontsize=13, fontweight='bold', pad=15)
axes[0].set_xlabel('Works Overtime?', fontsize=11)
axes[0].set_ylabel('Attrition Percentage (%)', fontsize=11)
axes[0].set_ylim(0, 40) # Add scale ceiling for a clean layout look

# --- PLOT 4 (Right): Work-Life Balance Score vs Attrition ---
sns.barplot(
    data=plot_wlb_compare,
    x='WorkLifeBalance',
    y='Attrition_Rate',
    ax=axes[1],
    palette='magma',
    hue='WorkLifeBalance',
    legend=False
)
axes[1].set_title('4. Attrition Rate by Work-Life Balance Score (1-Low, 4-High)', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('Work-Life Balance Score', fontsize=11)
axes[1].set_ylabel('Attrition Percentage (%)', fontsize=11)
axes[1].set_ylim(0, 40) # Match the scale ceiling of the left plot

# Prevent title/label overlaps and display the second master chart
plt.tight_layout()
plt.savefig('overtime_worklife_impact.png', dpi=300, bbox_inches='tight')
plt.show()


## Phase 6 — Executive Summary & Strategic Recommendations

*Based on our data calculations and visualizations across 1,470 employee records, we present our core strategic suggestions to help the CHRO reduce employee attrition and protect corporate capital.*

### 🏢 Executive Strategy Blueprint

#### 1. Compensation & Salary Strategy
* **Benchmark Entry-Level Pay:** Employees who left had an average monthly salary of **₹4,787**, compared to **₹6,832** for those who stayed. Benchmark and adjust base pay under ₹5,000 to remain market-competitive.
* **Align Hikes with Tenure:** Targeted salary reviews for early-career hires to reduce high vulnerability in lower-pay bands.

#### 2. Work-Life Balance Strategy
* **Address Stressed Cohorts:** Employees rating their Work-Life Balance as "1" (Poor) exhibit a critical **31.2% attrition rate**.
* **Introduce Flexible Models:** Pilot flexible working arrangements or hybrid structures to lift satisfaction scores to the stable "3" (Good) range, where attrition drops cleanly to **14.2%**.

#### 3. Overtime Cap Policy
* **Control the Overtime Trap:** Regular overtime is the single biggest predictor of attrition, driving a **30.5% resignation rate** compared to **10.4%** for standard hours.
* **Enforce Rest Limits:** Introduce strict weekly limits on maximum overtime hours, particularly in operational roles, to prevent burnout.

#### 4. Department-Specific Action Plan
* **Target the Sales Hotspot:** Sales leads all departments with a **20.6% attrition rate**, followed closely by Human Resources at **19.0%**.
* **Departmental Interventions:** Pilot burnout audits and employee-retention workshops specifically within Sales and HR teams.

### 🎯 Final Recommendation:
To protect our corporate capital from a staggering **₹22.5 Crore ($2.7M approx)** annualized replacement loss, the CHRO must immediately implement an **Overtime Cap Policy** in Sales/HR and execute a **Compensation Audit** for lower-tier salary bands under ₹5,000.
